title: "Interest Rates"
author: "Christian Kruse"
date: "`r Sys.Date()`"
output: html_document

In [ ]:
setwd(".")
getwd()

In [ ]:
knitr::opts_chunk$set(echo = FALSE,message = FALSE,warning = FALSE)
options(scipen=999)

In [ ]:
library(renv)
renv::init()

In [ ]:
install.packages(
  "dkstat",
  repos = c(
    ropengov = "https://ropengov.r-universe.dev",
    getOption("repos")
  )
)

In [ ]:
if (!require(pacman)) { install.packages("pacman") }
pacman::p_load(fredr,
               dplyr,
               tidyr,
               lubridate,
               caret,
               partykit,
               ggplot2,
               pROC,
            #    gganimate,
               devtools,
               httr,
               DT,
               ecb,
               httr,
               visNetwork,
               pbmcapply,
               dkstat,
               data.table,
               glue,
               openxlsx,
               pROC,
               rPref,
               progress,
               PlayerRatings,
               gganimate,
               quantmod,
               rvest,
               ggrepel)
source(file = "money_theme.R")

# Interest Rates

In [ ]:
# readRenviron(path = "Renviron.site")
# fredr::fredr_set_key(key = Sys.getenv("FRED_API"))

fredr::fredr_set_key(key = Sys.getenv("FRED_API"))

# test ECB

In [ ]:
ecb::get_data(key = glue("FM.D.U2.EUR.4F.KR.DFR.LEV"))

# normal

In [ ]:
df_yieldcurve = 
  fredr(series_id = "DGS1") %>% 
  bind_rows(fredr(series_id = "DGS2")) %>% 
  bind_rows(fredr(series_id = "DGS3")) %>% 
  bind_rows(fredr(series_id = "DGS5")) %>% 
  bind_rows(fredr(series_id = "DGS7")) %>% 
  bind_rows(fredr(series_id = "DGS10")) %>% 
  bind_rows(fredr(series_id = "DGS20")) %>% 
  bind_rows(fredr(series_id = "DGS30")) %>% 
  bind_rows(fredr(series_id = "DGS1MO")) %>% 
  bind_rows(fredr(series_id = "DGS3MO")) %>% 
  bind_rows(fredr(series_id = "DGS6MO")) %>% 
  bind_rows(fredr(series_id = "DFF")) %>% 
  bind_rows(fredr(series_id = "MORTGAGE30US"))

order_ = c("DFF",
           "DGS1MO",
           "DGS3MO",
           "DGS6MO",
           "DGS1",
           "DGS2",
           "DGS3",
           "DGS5",
           "DGS7",
           "DGS10",
           "DGS20",
           "DGS30",
           "MORTGAGE30US")

df_yieldcurve = 
  df_yieldcurve %>% 
  dplyr::select(date,series_id,value) %>% 
  dplyr::mutate(series_id=factor(series_id,levels=order_)) %>% 
  spread(series_id,value) %>% 
  arrange(desc(date)) %>% 
  tidyr::fill(c(2:ncol(.)),.direction="up") %>% 
  # na.omit() %>% 
  # top_n(n=1,wt=date) %>% 
  gather(series_id,value,c(2:ncol(.))) %>% 
  ungroup() %>% 
  dplyr::mutate(series_id=factor(series_id,levels=order_)) %>% 
  dplyr::mutate(time=ifelse(series_id=="DFF",0,
                            ifelse(substr(series_id,1,8)=="MORTGAGE",
                                   365*30,
                                   ifelse(substr(series_id,5,6)=="MO",
                                          30*as.integer(substr(series_id,4,4)),
                                          365*as.integer(substr(series_id,4,5)))))) 

## Treasury Bonds

In [ ]:
unloadNamespace("randomForest")
df_yieldcurve %>% 
  dplyr::mutate(value=value/100) %>% 
  ggplot(.,aes(x=date,y=value)) +
  geom_line(aes(group=series_id,color=series_id)) +
  scale_x_date(date_breaks = "5 year",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.1,1,by=0.025)) + 
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="All Yields",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

## Since 2001

In [ ]:
df_yieldcurve %>% 
  filter(date>ymd("2001-01-01")) %>% 
  dplyr::mutate(value=value/100) %>% 
  ggplot(.,aes(x=date,y=value)) +
  geom_line(aes(group=series_id,color=series_id)) +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.1,1,by=0.005)) + 
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="All Yields",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

### Previous 12 Months

In [ ]:
df_yieldcurve %>% 
  filter(date>=Sys.Date()-years(1)) %>% 
  dplyr::mutate(value=value/100) %>% 
  ggplot(.,aes(x=date,y=value)) +
  geom_line(aes(group=series_id,color=series_id)) +
  theme(legend.position="bottom") +
  scale_x_date(date_breaks = "1 months",date_labels = "%Y %b") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.1,1,by=0.005)) + 
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="All Yields (Last Year)",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

## Yield Curve

In [ ]:
df_yieldcurve %>% 
  dplyr::mutate(series_id=as.integer(paste0(recode(series_id,
                               'FEDFUNDS'='0',
                               'DGS1MO'='1',
                               'DGS3MO'='3',
                               'DGS6MO'='6',
                               'DGS1'='12',
                               'DGS2'='24',
                               'DGS3'='36',
                               'DGS5'='60',
                               'DGS7'='84',
                               'DGS10'='120',
                               'DGS20'='240',
                               'DGS30'='360')))) %>% 
  filter(date == max(date)) %>% 
  ggplot(.,aes(x=series_id,y=value)) +
  geom_point() +
  geom_line() +
  scale_x_continuous(labels = c('FEDFUNDS','DGS1MO','DGS3MO','DGS6MO','DGS1','DGS2','DGS3','DGS5','DGS7','DGS10','DGS20','DGS30'),breaks = c(0,1,3,6,12,24,36,60,84,120,240,360)) +
  scale_y_continuous(limits=c(0,NA),expand=c(0,0),breaks = seq(0,10,by=0.25))  + 
  # geom_text(aes(label=paste0(value,"%")),vjust=-1,size=3) +
  geom_text_repel(aes(label=paste0(value,"%")),size=3) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="US Treasury Yield Curve",
       x="Bond",
       y="Yield (%)",
       caption = timestamp_caption())

## Treasury Bonds Spreads

### DFF to All

In [ ]:
df_yieldcurve %>% 
  filter(date>ymd("1975-01-01")) %>% 
  dplyr::select(-time) %>% 
  # filter(series_id %in% c("DGS10","DGS2")) %>% 
  spread(series_id,value) %>% 
  gather(series_id,value,DGS1MO:MORTGAGE30US) %>% 
  dplyr::mutate(spread=(value-DFF)/100) %>% 
  ggplot(.,aes(x=date,y=spread,color=series_id)) +
  geom_line() +
  geom_hline(yintercept = 0) + 
  scale_x_date(date_breaks = "2 year",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent) + 
  theme_money_printer_go_brrr(base_size=12) +
  geom_hline(yintercept = c(-0.0025,-0.005,-0.0075,-0.01),linetype=2,color="gray60") +
  labs(title="Spread: DFF to ALL",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

In [ ]:
df_yieldcurve %>% 
  filter(date>Sys.Date()-years(1)) %>% 
  dplyr::select(-time) %>% 
  # filter(series_id %in% c("DGS10","DGS2")) %>% 
  spread(series_id,value) %>% 
  gather(series_id,value,DGS1MO:MORTGAGE30US) %>% 
  dplyr::mutate(spread=(value-DFF)/100) %>% 
  ggplot(.,aes(x=date,y=spread,color=series_id)) +
  geom_line() +
  geom_hline(yintercept = 0) + 
  scale_x_date(date_breaks = "2 months",date_labels = "%Y-%b") +
  scale_y_continuous(labels = scales::percent) + 
  theme_money_printer_go_brrr(base_size=12) +
  geom_hline(yintercept = c(-0.0025,-0.005,-0.0075,-0.01),linetype=2,color="gray60") +
  labs(title="Spread: DFF to ALL",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

### 2-Year to 10-Year T-Bond

In [ ]:
df_yieldcurve %>% 
  filter(date>ymd("1975-01-01")) %>% 
  dplyr::select(-time) %>% 
  filter(series_id %in% c("DGS10","DGS2")) %>% 
  spread(series_id,value) %>% 
  dplyr::mutate(spread=(DGS10-DGS2)/100) %>% 
  ggplot(.,aes(x=date,y=spread)) +
  geom_line() +
  geom_hline(yintercept = 0) + 
  scale_x_date(date_breaks = "2 year",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.1,1,by=0.0025)) + 
  theme_money_printer_go_brrr(base_size=12) +
  geom_hline(yintercept = c(-0.0025,-0.005,-0.0075,-0.01),linetype=2,color="gray60") +
  labs(title="Spread: DGS2 - DGS10",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

In [ ]:
df_yieldcurve %>% 
  filter(date>Sys.Date()-years(1)) %>% 
  dplyr::select(-time) %>% 
  filter(series_id %in% c("DGS10","DGS2")) %>% 
  spread(series_id,value) %>% 
  dplyr::mutate(spread=(DGS10-DGS2)/100) %>% 
  ggplot(.,aes(x=date,y=spread)) +
  geom_line() +
  geom_hline(yintercept = 0,linetype=2) + 
  scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.1,1,by=0.0025)) + 
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Spread: DGS2 - DGS10",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

### 30-Year Mortgage to 30-Year T-Bond

In [ ]:
df_yieldcurve %>% 
  filter(date>ymd("1975-01-01")) %>% 
  filter(series_id %in% c("MORTGAGE30US","DGS30")) %>% 
  spread(series_id,value) %>% 
  dplyr::mutate(spread=(MORTGAGE30US-DGS30)/100) %>% 
  ggplot(.,aes(x=date,y=spread)) +
  geom_line() +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.1,1,by=0.005)) +
  geom_hline(yintercept = 0,linetype=2) +
  geom_hline(yintercept = 0.025,color="red") +
  theme_money_printer_go_brrr(base_size=12) +
  theme(legend.position="bottom") +
  labs(title="Spread: Mortgage30US - DGS30",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

### 30-Year Mortgage to 10-Year T-Bond

In [ ]:
df_yieldcurve %>% 
  dplyr::select(-time) %>% 
  filter(date>ymd("1975-01-01")) %>% 
  filter(series_id %in% c("MORTGAGE30US","DGS10")) %>% 
  spread(series_id,value) %>% 
  dplyr::mutate(spread=(MORTGAGE30US-DGS10)/100) %>% 
  ggplot(.,aes(x=date,y=spread)) +
  geom_line() +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.1,1,by=0.005)) +
  geom_hline(yintercept = 0,linetype=2) +
  geom_hline(yintercept = 0.025,color="red") +
  theme_money_printer_go_brrr(base_size=12) +
  theme(legend.position="bottom") +
  labs(title="Spread: Mortgage30US - DGS10",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

## Yield Checkboard

In [ ]:
df_checkboard = expand.grid(series_id=unique(df_yieldcurve$series_id),
                            joined_series=unique(df_yieldcurve$series_id))

df_yield_checkboard = 
  df_checkboard %>% 
  inner_join(df_yieldcurve) %>% 
  inner_join(df_yieldcurve,by=c("joined_series"="series_id","date"="date")) %>% 
  dplyr::mutate(spread=value.x-value.y) %>% 
  dplyr::mutate(label=ifelse(spread<0,spread,NA)) %>% 
  dplyr::select(series_id,joined_series,date,spread,label) %>% 
  dplyr::mutate(series_id=factor(series_id,levels=order_),
                joined_series=factor(joined_series,levels=order_))

In [ ]:
df_yield_checkboard %>% 
  group_by(series_id,joined_series) %>% 
  filter(date==max(date)) %>% 
  ungroup() %>% 
  dplyr::mutate(spread=ifelse(series_id==joined_series,NA,spread)) %>% 
  ggplot(.,aes(x=series_id,y=joined_series)) +
  geom_tile(aes(fill=spread)) +
  scale_fill_gradient2(low="green",high="red",midpoint=0,na.value = "gray70") +
  geom_text(aes(label=scales::number(label,accuracy = 0.01,prefix = NULL)),size=3.5)+ 
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="US T-Bond Spreads Checkboard",
       x=NULL,
       y="NULL",
       caption = timestamp_caption())

In [ ]:
# Install plotly
library(plotly)

# Convert your ggplot to animated plotly
p <- df_yield_checkboard %>% 
  filter(date>ymd("20-01-01")) %>% 
  ggplot(.,aes(x=series_id,y=joined_series)) +
  geom_tile(aes(fill=spread, frame=date)) +
  scale_fill_gradient2(low="green",high="red",midpoint=0) +
  theme(axis.text.x=element_text(angle=45,hjust=1),
        legend.position="bottom")+
  geom_text(aes(label=scales::number(label,accuracy = 0.01,prefix = NULL)),size=3.5)

ggplotly(p) %>% 
  animation_opts(frame = 100, transition = 50)

In [ ]:
# df_yield_checkboard %>% 
#   filter(date>ymd("20-01-01")) %>% 
#   ggplot(.,aes(x=series_id,y=joined_series)) +
#   geom_tile(aes(fill=spread)) +
#   scale_fill_gradient2(low="green",high="red",midpoint=0) +
#   theme(axis.text.x=element_text(angle=45,hjust=1),
#         legend.position="bottom")+
#   geom_text(aes(label=scales::number(label,accuracy = 0.01,prefix = NULL)),size=3.5)  + 
#   transition_time(date) +
#   labs(title = "Date: {frame_time}")

In [ ]:
df_yield_checkboard %>% 
  dplyr::mutate(series_id=as.integer(paste0(recode(series_id,
                                 'DFF'='0',
                                 'DGS1MO'='1',
                                 'DGS3MO'='3',
                                 'DGS6MO'='6',
                                 'DGS1'='12',
                                 'DGS2'='24',
                                 'DGS3'='36',
                                 'DGS5'='60',
                                 'DGS7'='84',
                                 'DGS10'='120',
                                 'DGS20'='240',
                                 'DGS30'='360')))) %>% 
  dplyr::mutate(joined_series=as.integer(paste0(recode(joined_series,
                                 'DFF'='0',
                                 'DGS1MO'='1',
                                 'DGS3MO'='3',
                                 'DGS6MO'='6',
                                 'DGS1'='12',
                                 'DGS2'='24',
                                 'DGS3'='36',
                                 'DGS5'='60',
                                 'DGS7'='84',
                                 'DGS10'='120',
                                 'DGS20'='240',
                                 'DGS30'='360')))) %>% 
  dplyr::mutate(diff=series_id-joined_series) %>% 
  filter(series_id!=joined_series) %>% 
  filter(series_id > joined_series) %>% 
  filter(date>ymd("2022-01-01")) %>% 
  dplyr::mutate(series_id=recode(series_id,
                                 '0'='DFF',
                                 '1'='DGS1MO',
                                 '3'='DGS3MO',
                                 '6'='DGS6MO',
                                 '12'='DGS1',
                                 '24'='DGS2',
                                 '36'='DGS3',
                                 '60'='DGS5',
                                 '84'='DGS7',
                                 '120'='DGS10',
                                 '240'='DGS20',
                                 '360'='DGS30')) %>% 
  dplyr::mutate(joined_series=recode(joined_series,
                                 '0'='DFF',
                                 '1'='DGS1MO',
                                 '3'='DGS3MO',
                                 '6'='DGS6MO',
                                 '12'='DGS1',
                                 '24'='DGS2',
                                 '36'='DGS3',
                                 '60'='DGS5',
                                 '84'='DGS7',
                                 '120'='DGS10',
                                 '240'='DGS20',
                                 '360'='DGS30')) %>% 
  unite(series_id,series_id,joined_series) %>% 
  group_by(series_id) %>% 
  dplyr::mutate(label=ifelse(date==max(date),paste0(series_id),"")) %>% 
  ggplot(.,aes(x=date,y=spread,color=series_id)) +
  geom_line()+
  theme_bw() +
  theme(legend.position="null") +
  scale_y_continuous(limits=c(NA,0.25)) +
  scale_x_date(expand=c(0.4,0)) +
  geom_text(aes(label=label),hjust=-0.1,size=3) +
  geom_hline(yintercept = 0,linetype=2) +
  geom_vline(xintercept = Sys.Date())+ 
  theme_money_printer_go_brrr(base_size=12) +
  theme(legend.position="null") +
  labs(title="US T-Bond Spread Over Time",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

# Denmark Mortgage Bonds

In [ ]:
fit_draw_abline = function(date1,date2,value1,value2) {
  
  data.frame(Date=c(date1,date2),
             val=c(value1,value2)) %>% 
    lm(data = .,formula = val~Date) %>% 
    coef(.) %>% 
    as.numeric(.) %>% 
    geom_abline(intercept = .[1],slope = .[2])
  
}

In [ ]:
source_url = "https://finansdanmark.dk/tal-og-data/boligstatistik/obligationsrenter/"
current_url = xml2::read_html(source_url) %>%
  html_node("body > main > div > div.page-header > div.page-header__content > div > div.row > div.col-12.col-md-8 > div > p:nth-child(9) > strong > span > a") %>% 
  rvest::html_attr("href")
xlsx_url = paste0("https://finansdanmark.dk/",current_url)

In [ ]:
df_interest = openxlsx::read.xlsx(xlsxFile = xlsx_url,startRow = 1)
df_interest$År[1] = 1997
df_interest = df_interest %>% fill(År,.direction = "down")
df_interest$Date = as.Date(paste(df_interest$År, df_interest$Uge, 1, sep="-"), "%Y-%U-%u")
df_interest = df_interest %>% 
  dplyr::select(Date,Kort.rente,Lang.rente) %>% 
  dplyr::mutate( CurveLongShort = Lang.rente - Kort.rente )

max_date = max(df_interest$Date,na.rm = T)

## Short, Long + Spread

In [ ]:
df_interest %>% 
  gather(stat,val,Kort.rente:Lang.rente) %>% 
  dplyr::mutate(stat=recode(stat,"Kort.rente"="Short","Lang.rente"="30yr","CurveLongShort"="Spread")) %>% 
  dplyr::mutate(val=val/100) %>% 
  ggplot(.,aes(x=Date,y=val,color=stat)) +
  geom_line() +
  scale_x_date(date_breaks = "2 years",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.03,0.1,by=0.005)) +
  # geom_vline(data=data.frame(Date=c(ymd("2005-08-05"),ymd("2005-09-15"),ymd("2021-04-05"),ymd("2021-06-14"))),aes(xintercept=Date)) + 
  geom_hline(yintercept = 0,linetype=2) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Danish Long and Short Interest Rate",
       subtitle=paste0("Source: ",source_url),
       x="Date",
       y="(%)",
       caption = paste0("Data from ",format(max_date,"%Y %b %d"),". ",timestamp_caption())) +
  geom_vline(xintercept = max_date)

### Previous 12 Months

In [ ]:
df_interest %>% 
  filter(Date>=Sys.Date()-years(1)) %>%  
  gather(stat,val,Kort.rente:CurveLongShort) %>% 
  dplyr::mutate(stat=recode(stat,"Kort.rente"="Short","Lang.rente"="30yr","CurveLongShort"="Spread")) %>% 
  dplyr::mutate(val=val/100) %>% 
  ggplot(.,aes(x=Date,y=val,color=stat)) +
  geom_line() +
  scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.03,0.1,by=0.005)) +
  geom_hline(yintercept = 0,linetype=2) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Danish Long and Short Interest Rate",
       subtitle=paste0("Source: ",source_url),
       x="Date",
       y="(%)",
       caption = paste0("Data from ",format(max_date,"%Y %b %d"),". ",timestamp_caption())) +
  geom_vline(xintercept = max_date)

### Spread

In [ ]:
df_interest %>% 
  gather(stat,val,CurveLongShort) %>% 
  dplyr::mutate(stat=recode(stat,"Kort.rente"="Short","Lang.rente"="30yr","CurveLongShort"="Spread")) %>% 
  dplyr::mutate(val=val/100) %>% 
  ggplot(.,aes(x=Date,y=val,color=stat)) +
  geom_line() +
  scale_x_date(date_breaks = "1 years",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.03,0.1,by=0.005)) +
  # geom_vline(data=data.frame(Date=c(ymd("2005-08-05"),ymd("2005-09-15"),ymd("2021-04-05"),ymd("2021-06-14"))),aes(xintercept=Date)) + 
  # geom_hline(yintercept = 0,linetype=2) +
  geom_hline(yintercept = 0.035,color="red") +
  # fit_draw_abline(date1 = ymd("2021-01-01"),date2 = ymd("2022-09-01"),value1 = 0.015,value2 = 0.035) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Spread: Danish Long and Short Interest Rate",
       subtitle=paste0("Source: ",source_url),
       x="Date",
       y="(%)",
       caption = paste0("Data from ",format(max_date,"%Y %b %d"),". ",timestamp_caption()))

### Short w/ ECB Rate Changes

In [ ]:
df_interest %>% 
  dplyr::mutate(Kort.rente=Kort.rente/100) %>% 
  ggplot(.,aes(x=Date,y=Kort.rente)) +
  geom_line() +
  scale_y_continuous(limits=c(-0.01,NA),breaks = seq(-0.01,0.07,by=0.005),labels = scales::percent) +
  geom_vline(xintercept = dmy("01-12-2018")) +
  geom_vline(xintercept = c(dmy("09-11-2011"),
                            dmy("14-12-2011"),
                            dmy("11-07-2012"),
                            dmy("13-11-2013"),
                            dmy("08-05-2013"),
                            dmy("11-06-2014"),
                            dmy("10-09-2014"),
                            dmy("09-12-2015"),
                            dmy("16-03-2016"),
                            dmy("18-09-2019")),color="blue",linetype=2) +
  geom_vline(xintercept = c(dmy("13-04-2011"),
                            dmy("13-07-2011")),color="red",linetype=2) + 
  geom_hline(yintercept = 0,linetype=2) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Short Yield",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption())

## Parabolic Support and Resistance

In [ ]:
# Decline
lm_low = data.frame(Date=c(ymd("2010-08-30","2019-09-09")),
           Lang.Rente=c(4.12000,0.68330)) %>% 
  lm(log(Lang.Rente)~Date,data=.)

lm_high = data.frame(Date=c(ymd("2011-04-01","2015-07-01")),
           Lang.Rente=c(5.3,3.4)) %>% 
  lm(log(Lang.Rente)~Date,data=.)

# Surge
lm_low_2 = data.frame(Date=c(ymd("2022-07-30","2022-11-09")),
           Lang.Rente=c(3.3,4.6)) %>% 
  lm(log(Lang.Rente)~Date,data=.)

lm_high_2 = data.frame(Date=c(ymd("2022-06-01","2021-07-01")),
           Lang.Rente=c(4.2,1.8)) %>% 
  lm(log(Lang.Rente)~Date,data=.)

# Graph
df_interest %>% 
  filter(Date>=ymd("2010-01-01")) %>%  
  dplyr::mutate(Lang.rente=Lang.rente/100) %>% 
  dplyr::mutate(Pred_High=exp(predict(lm_high,.))/100,
                Pred_Low=exp(predict(lm_low,.))/100) %>% 
  dplyr::mutate(Pred_High_2=exp(predict(lm_high_2,.))/100,
                Pred_Low_2=exp(predict(lm_low_2,.))/100) %>% 
  ggplot(.,aes(x=Date,y=Lang.rente)) +
  geom_line() +
  geom_line(aes(y=Pred_High),linetype=2) +
  geom_line(aes(y=Pred_Low),linetype=2) +
  geom_line(aes(y=Pred_High_2),linetype=2) +
  geom_line(aes(y=Pred_Low_2),linetype=2) +
  scale_y_continuous(limits=c(0,0.065),breaks = seq(0,0.065,by=0.005),labels = scales::percent) +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  geom_hline(yintercept = 0,linetype=2) +
  geom_vline(xintercept = dmy("01-12-2018")) +
  geom_vline(xintercept = c(dmy("09-11-2011"),
                            dmy("14-12-2011"),
                            dmy("11-07-2012"),
                            dmy("13-11-2013"),
                            dmy("08-05-2013"),
                            dmy("11-06-2014"),
                            dmy("10-09-2014"),
                            dmy("09-12-2015"),
                            dmy("16-03-2016"),
                            dmy("18-09-2019")),color="blue",linetype=2) +
  geom_vline(xintercept = c(dmy("13-04-2011"),
                            dmy("13-07-2011")),color="red",linetype=2) +
  geom_vline(xintercept = c(dmy("01-02-2017"),
                            dmy("01-10-2017"),
                            dmy("01-12-2018")),color="green",linetype=2) + 
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="All Yields (Last Year)",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption()) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Exponential Boundaries of 30yr Mortgage Rate (w/ EBC Rate Changes)",
       subtitle=paste0("Source: ",source_url),
       x="Date",
       y="(%)",
       caption = timestamp_caption())

## Historic 5-Year Forward Rate Increases

In [ ]:
df_interest %>% 
  dplyr::select(Date,Kort.rente,Lang.rente) %>% 
  gather(stat,val,Kort.rente:Lang.rente) %>% 
  group_by(stat) %>% 
  # arrange(stat,desc(Date)) %>% 
  arrange(stat,Date) %>% 
  dplyr::mutate(FutureMax=rollmax(val,k = 5*52,fill = NA,align = "left")-val) %>% 
  ggplot(.,aes(x=Date,y=FutureMax,color=stat)) +
  geom_line() +
  scale_x_date(date_breaks = "1 years",date_labels = "%Y") +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Future Rate Increases within 5 yrs",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption()) 

In [ ]:
df_interest %>% 
  dplyr::select(Date,Kort.rente,Lang.rente) %>% 
  gather(stat,val,Kort.rente:Lang.rente) %>% 
  group_by(stat) %>% 
  # arrange(stat,desc(Date)) %>% 
  arrange(stat,Date) %>% 
  dplyr::mutate(FutureMax=rollmax(val,k = 5*52,fill = NA,align = "left")-val) %>% 
  ggplot(.,aes(x=Date,y=FutureMax,color=stat)) +
  geom_line() +
  scale_x_date(date_breaks = "1 years",date_labels = "%Y") +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Future Rate Increases within 5 yrs",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption()) +
  facet_wrap(~stat)

## Lowest Long Yield and Smallest Spread

In [ ]:
df_latest = df_interest %>% 
  dplyr::mutate(Lang.rente=Lang.rente/100,
                CurveLongShort=CurveLongShort/100) %>% 
  filter(Date==max(Date,na.rm=T)) %>% 
  dplyr::mutate(Time_Ago=as.numeric(Sys.Date()-Date)) 


df_interest %>% 
  dplyr::mutate(Lang.rente=Lang.rente/100,
                CurveLongShort=CurveLongShort/100) %>% 
  dplyr::mutate(Time_Ago=as.numeric(Sys.Date()-Date)) %>% 
  ggplot(.,aes(y=Lang.rente,x=CurveLongShort,color=Time_Ago)) +
  geom_point() +
  scale_y_continuous(limits=c(0,NA),breaks = seq(0,0.1,by=0.005),labels = scales::percent) +
  scale_x_continuous(limits=c(0,NA),breaks = seq(0,0.1,by=0.005),labels = scales::percent) + 
  geom_text_repel(data=df_latest,aes(label=scales::percent(Lang.rente,accuracy = 0.1)),nudge_x = -0.01,nudge_y=-0.01) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Lowest Long Yield to Lowest Short-Long Spread",
       y="Mortgage 30YR Yield",
       x="Short-Long Spread",
       caption = timestamp_caption())

## Best Time to Go Fixed

In [ ]:
df_glicko = data.frame(t(combn(df_interest %>% filter(Date>ymd("2001-01-01")) %>% pull(Date),2)))
df_glicko$X1 = as.Date(df_glicko$X1,origin="1970-01-01")
df_glicko$X2 = as.Date(df_glicko$X2,origin="1970-01-01")

df_glicko_ratings = 
  df_glicko %>% 
  inner_join(df_interest,by=c("X1"="Date")) %>% 
  inner_join(df_interest,by=c("X2"="Date")) %>% 
  dplyr::mutate(Score1=ifelse(Lang.rente.x<Lang.rente.y,1,ifelse(Lang.rente.x>Lang.rente.y,-1,0))) %>% 
  dplyr::mutate(Score2=ifelse(CurveLongShort.x<CurveLongShort.y,1,ifelse(CurveLongShort.x>CurveLongShort.y,-1,0))) %>% 
  dplyr::mutate(Score=Score1+Score2) %>% 
  dplyr::select(X1,X2,Score) %>% 
  dplyr::mutate(result=ifelse(Score>0,1,
                              ifelse(Score<0,0,0.5))) %>% 
  na.omit() %>% 
  dplyr::mutate(Time=1,
                X1=paste0(X1),
                X2=paste0(X2)) %>% 
  dplyr::select(Time,X1,X2,result) %>% 
  PlayerRatings::glicko(.)  %>% 
  .[["ratings"]] %>% 
  dplyr::mutate(Rating=scales::rescale(Rating,to=c(0,100)))

In [ ]:
df_glicko_ratings %>% 
  dplyr::mutate(Player=ymd(Player)) %>% 
  ggplot(.,aes(x=Player,y=Rating)) +
  geom_line() +
  scale_y_continuous(limits=c(0,100),breaks = seq(0,100,by=10)) +
  scale_x_date(date_breaks = "1 years",date_labels = "%Y") +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Best Time To Go Fixed (0-100)",
       x="Year",
       y="Rating (0-100)",
       caption = timestamp_caption())

In [ ]:
df_glicko_ratings %>% 
  filter(Player>=Sys.Date()-years(2)) %>% 
  dplyr::mutate(Player=ymd(Player)) %>% 
  ggplot(.,aes(x=Player,y=Rating)) +
  geom_line() +
  scale_y_continuous(limits=c(0,100),breaks = seq(0,100,by=10)) +
  scale_x_date(date_breaks = "2 months",date_labels = "%Y") +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Best Time To Go Fixed (0-100)",
       x="Year",
       y="Rating (0-100)",
       caption = timestamp_caption())

# Predict US Interest Rates

In [ ]:
fomc_dates = data.frame(date_fomc=c(ymd("2022-06-15"),
                            ymd("2022-07-27"),
                            ymd("2022-09-21"),
                            ymd("2022-11-02"),
                            ymd("2022-12-14"),
                            ymd("2023-02-01"),
                            ymd("2023-03-15"),
                            ymd("2023-05-03"),
                            ymd("2023-06-14"),
                            ymd("2023-07-26"),
                            ymd("2023-09-20"),
                            ymd("2023-11-01"),
                            ymd("2023-12-13"),
                            ymd("2024-01-31"),
                            ymd("2024-03-20"),
                            ymd("2024-05-01"),
                            ymd("2024-06-19"),
                            ymd("2024-07-31"),
                            ymd("2024-09-25"),
                            ymd("2024-11-06"),
                            ymd("2024-12-18"),
                            Sys.Date())) %>% 
  filter(date_fomc>=Sys.Date())

add_fomc_meeting_dates = function() {
  geom_vline(xintercept = fomc_dates$date_fomc,linetype=2,color="#eb493a")
}

In [ ]:
num_trees=100
tune_grid=data.frame(mtry=c(3))

In [ ]:
df_yieldcurve = 
  fredr(series_id = "DGS1") %>% 
  bind_rows(fredr(series_id = "DGS2")) %>% 
  bind_rows(fredr(series_id = "DGS3")) %>% 
  bind_rows(fredr(series_id = "DGS5")) %>% 
  bind_rows(fredr(series_id = "DGS7")) %>% 
  bind_rows(fredr(series_id = "DGS10")) %>% 
  bind_rows(fredr(series_id = "DGS20")) %>% 
  bind_rows(fredr(series_id = "DGS30")) %>% 
  bind_rows(fredr(series_id = "DGS1MO")) %>% 
  bind_rows(fredr(series_id = "DGS3MO")) %>% 
  bind_rows(fredr(series_id = "DGS6MO")) %>% 
  bind_rows(fredr(series_id = "DFF")) %>% 
  bind_rows(fredr(series_id = "MORTGAGE30US"))

order_ = c("DFF",
                                                    "DGS1MO",
                                                    "DGS3MO",
                                                    "DGS6MO",
                                                    "DGS1",
                                                    "DGS2",
                                                    "DGS3",
                                                    "DGS5",
                                                    "DGS7",
                                                    "DGS10",
                                                    "DGS20",
                                                    "DGS30",
                                                    "MORTGAGE30US")
df_yieldcurve = 
  df_yieldcurve %>% 
  dplyr::select(date,series_id,value) %>% 
  dplyr::mutate(series_id=factor(series_id,levels=order_)) %>% 
  spread(series_id,value) %>% 
  arrange(desc(date)) %>% 
  tidyr::fill(c(2:ncol(.)),.direction="up") %>% 
  na.omit() %>% 
  # top_n(n=1,wt=date) %>% 
  gather(series_id,value,c(2:ncol(.))) %>% 
  ungroup() %>% 
  dplyr::mutate(series_id=factor(series_id,levels=order_)) %>% 
  dplyr::mutate(time=ifelse(series_id=="DFF",0,
                            ifelse(substr(series_id,1,8)=="MORTGAGE",
                                   365*30,
                                   ifelse(substr(series_id,5,6)=="MO",
                                          30*as.integer(substr(series_id,4,4)),
                                          365*as.integer(substr(series_id,4,5)))))) 

In [ ]:
source_url = "https://finansdanmark.dk/tal-og-data/boligstatistik/obligationsrenter/"
current_url = xml2::read_html(source_url) %>%
  html_node("body > main > div > div.page-header > div.page-header__content > div > div.row > div.col-12.col-md-8 > div > p:nth-child(9) > strong > span > a") %>% 
  rvest::html_attr("href")
xlsx_url = paste0("https://finansdanmark.dk/",current_url)

df_DK = openxlsx::read.xlsx(xlsxFile = xlsx_url,startRow = 1)


df_DK$År[1] = 1997
df_DK = df_DK %>% fill(År,.direction = "down")

df_DK$date = as.Date(paste(df_DK$År, df_DK$Uge, 1, sep="-"), "%Y-%U-%u")
df_DK = df_DK %>% 
  dplyr::select(date,Kort.rente,Lang.rente) %>% 
  rename(DK_Short=Kort.rente,
         DK_Long=Lang.rente) %>% 
  dplyr::mutate( DK_Spread = DK_Short - DK_Long ) %>% 
  gather(series_id,value,DK_Short:DK_Spread)

# df_interest$Date = as.Date(paste(df_interest$År, df_interest$Uge, 1, sep="-"), "%Y-%U-%u")
# df_interest = df_interest %>% 
#   dplyr::select(Date,Kort.rente,Lang.rente) %>% 
#   dplyr::mutate( CurveLongShort = Lang.rente - Kort.rente )

# max_date = max(df_interest$Date,na.rm = T)

In [ ]:
df_INFLATION = 
  fredr(series_id = "CPIAUCSL") %>% 
  arrange(date) %>% 
  dplyr::mutate(value=100*((value/lag(x = value,n = 12))-1))
df_CSUSHPINSA = 
  fredr(series_id = "CSUSHPINSA") %>% 
  arrange(date)
df_M2SL = 
  fredr(series_id = "M2SL") %>% 
  arrange(date)
df_UMCSENT = 
  fredr(series_id = "UMCSENT") %>% 
  arrange(date)
df_USREC = 
  fredr(series_id = "USREC") %>% 
  arrange(date)

In [ ]:
# df_checkboard = expand.grid(series_id=unique(df_yieldcurve$series_id),
#                             joined_series=unique(df_yieldcurve$series_id))
df_checkboard = data.frame(t(data.frame(combn(unique(df_yieldcurve$series_id), 2, simplify = FALSE))),row.names = NULL)
colnames(df_checkboard) = c("series_id","joined_series")

df_yield_checkboard = 
  df_checkboard %>% 
  inner_join(df_yieldcurve) %>% 
  inner_join(df_yieldcurve,by=c("joined_series"="series_id","date"="date")) %>% 
  dplyr::mutate(spread=value.x-value.y) %>% 
  dplyr::mutate(label=ifelse(spread<0,spread,NA)) %>% 
  dplyr::select(series_id,joined_series,date,spread,label) %>% 
  dplyr::mutate(series_id=factor(series_id,levels=order_),
                joined_series=factor(joined_series,levels=order_))

In [ ]:
train_control = caret::trainControl(method = "repeatedcv",number = 5,repeats = 5)

In [ ]:
df_DFF = 
  df_yieldcurve %>% 
  filter(series_id=="DFF") %>% 
  dplyr::select(date,value)

df_DGS2 = 
  df_yieldcurve %>% 
  filter(series_id=="DGS2") %>% 
  dplyr::select(date,value)

df_DGS10 = 
  df_yieldcurve %>% 
  filter(series_id=="DGS10") %>% 
  dplyr::select(date,value)

df_MORTGAGE30US = 
  df_yieldcurve %>% 
  filter(series_id=="MORTGAGE30US") %>% 
  dplyr::select(date,value)

df_DK_Short =
  df_DK %>% 
  filter(series_id=="DK_Short") %>% 
  dplyr::select(date,value) %>% 
  tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=max(date,na.rm=T),by="1 day")) %>% 
  tidyr::fill(value,.direction = "down")

df_DK_Long = 
  df_DK %>% 
  filter(series_id=="DK_Long") %>% 
  dplyr::select(date,value) %>% 
  tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=max(date,na.rm=T),by="1 day")) %>% 
  tidyr::fill(value,.direction = "down")

df_CPIAUCSL = 
  df_INFLATION %>% 
  filter(series_id=="CPIAUCSL") %>% 
  dplyr::select(date,value) %>% 
  tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=max(date,na.rm=T),by="1 day")) %>% 
  tidyr::fill(value,.direction = "down")

df_CSUSHPINSA = 
  df_CSUSHPINSA %>% 
  filter(series_id=="CSUSHPINSA") %>% 
  dplyr::select(date,value) %>% 
  tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=max(date,na.rm=T),by="1 day")) %>% 
  tidyr::fill(value,.direction = "down")

df_M2SL = 
  df_M2SL %>% 
  filter(series_id=="M2SL") %>% 
  dplyr::select(date,value) %>% 
  tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=max(date,na.rm=T),by="1 day")) %>% 
  tidyr::fill(value,.direction = "down")

df_UMCSENT = 
  df_UMCSENT %>% 
  filter(series_id=="UMCSENT") %>% 
  dplyr::select(date,value) %>% 
  tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=max(date,na.rm=T),by="1 day")) %>% 
  tidyr::fill(value,.direction = "down")

df_USREC = 
  df_USREC %>% 
  filter(series_id=="USREC") %>% 
  dplyr::select(date,value) %>% 
  tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=max(date,na.rm=T),by="1 day")) %>% 
  tidyr::fill(value,.direction = "down")

In [ ]:
df_predict_DFF = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_DFF,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread)

df_predict_DGS2 = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_DGS2,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread)

df_predict_DGS10 = df_yield_checkboard %>%
  filter(series_id!=joined_series) %>%
  dplyr::mutate(one_year_ahead=date+years(1)) %>%
  left_join(df_DGS10,by=c("one_year_ahead"="date")) %>%
  unite(series_id,series_id,joined_series) %>%
  dplyr::select(date,series_id,spread,value) %>%
  spread(series_id,spread)

df_predict_MORTGAGE30US = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_MORTGAGE30US,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread)

df_predict_DK_Short = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_DK_Short,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread) 

df_predict_DK_Long = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_DK_Long,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread) %>% 
  arrange(desc(date))

df_predict_DK_Long = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_DK_Long,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread) %>% 
  arrange(desc(date))

df_predict_CPIAUCSL = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_CPIAUCSL,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread) %>% 
  arrange(desc(date))


df_predict_CSUSHPINSA = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_CSUSHPINSA,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread) %>% 
  arrange(desc(date))
df_predict_M2SL = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_M2SL,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread) %>% 
  arrange(desc(date))


df_predict_UMCSENT = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_UMCSENT,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread) %>% 
  arrange(desc(date))
df_predict_USREC = df_yield_checkboard %>% 
  filter(series_id!=joined_series) %>% 
  dplyr::mutate(one_year_ahead=date+years(1)) %>% 
  left_join(df_USREC,by=c("one_year_ahead"="date")) %>% 
  unite(series_id,series_id,joined_series) %>% 
  dplyr::select(date,series_id,spread,value) %>% 
  spread(series_id,spread) %>% 
  arrange(desc(date))

In [ ]:
# Load parallel processing libraries with progress bar
library(pbmcapply)

# Detect number of cores
num_cores <- detectCores() - 1  # Leave one core free

df_combinations_split = split(expand.grid(mtry=seq(15),num_trees=c(10,100,1000)),
                              seq(nrow(expand.grid(mtry=seq(15),
            num_trees=c(10,100,1000))))) 

# Use pbmclapply for parallel processing with progress bar
df_results = do.call("rbind", pbmclapply(df_combinations_split, function(row_) {
  # Load required libraries within each worker
  library(caret)
  library(dplyr)
  library(randomForest)
  
  # Make sure the data is available in the worker environment
  # (passed as arguments or ensure it's in the global environment)
  
  caret::train(x = df_predict_DFF %>% 
                              na.omit() %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_DFF %>% 
                           na.omit() %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=data.frame(mtry=row_$mtry),
                         ntree=row_$num_trees,
                         metric = "RMSE",
                         trControl=train_control)$results
}, mc.cores = num_cores))

In [ ]:
pacman::p_load(randomForest)
# library(randomForest)
fit_DFF = caret::train(x = df_predict_DFF %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_DFF %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_DGS2 = caret::train(x = df_predict_DGS2 %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_DGS2 %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_DGS10= caret::train(x = df_predict_DGS10 %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_DGS10 %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_MORTGAGE30US = caret::train(x = df_predict_MORTGAGE30US %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_MORTGAGE30US %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_DK_Short = caret::train(x = df_predict_DK_Short %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_DK_Short %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_DK_Long = caret::train(x = df_predict_DK_Long %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_DK_Long %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_CPIAUCSL = caret::train(x = df_predict_CPIAUCSL %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_CPIAUCSL %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_CSUSHPINSA = caret::train(x = df_predict_CSUSHPINSA %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_CSUSHPINSA %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_M2SL = caret::train(x = df_predict_M2SL %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_M2SL %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_UMCSENT = caret::train(x = df_predict_UMCSENT %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_UMCSENT %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_USREC = caret::train(x = df_predict_USREC %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = df_predict_USREC %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)

detach(package:randomForest,unload=TRUE)

## Predicted US Yield

In [ ]:
df_predict_DFF %>% 
  dplyr::mutate(pred_DFF=predict(fit_DFF,.)) %>% 
  dplyr::select(date,pred_DFF) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  filter(date>=ymd("2000-01-01")) %>% 
  gather(pred,val,pred_DFF) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>%
  filter(!one_year) %>% 
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line(size=0.25) +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  scale_y_continuous(breaks = seq(0,10,0.5)) +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  # facet_wrap(~one_year,scales="free",ncol=1) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted/Observed T-Bond Yields",
       x=NULL,
       y="(%)",
       caption = timestamp_caption()) +
  add_fomc_meeting_dates()

In [ ]:
df_predict_DFF %>% 
  dplyr::mutate(pred_DFF=predict(fit_DFF,.),
                pred_DGS2=predict(fit_DGS2,.),
                pred_DGS10=predict(fit_DGS10,.),
                pred_MORTGAGE30US=predict(fit_MORTGAGE30US,.)) %>% 
  dplyr::select(date,pred_DFF:pred_MORTGAGE30US) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_DFF:pred_MORTGAGE30US) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>%
  filter(!one_year) %>% 
  filter(date>=ymd("2000-01-01")) %>% 
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line(size=0.25) +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  scale_y_continuous(breaks = seq(0,10,0.5)) +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  # facet_wrap(~one_year,scales="free",ncol=1) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted/Observed T-Bond Yields",
       x=NULL,
       y="(%)",
       caption = timestamp_caption()) +
  add_fomc_meeting_dates()

In [ ]:
df_predict_DFF %>% 
  dplyr::mutate(pred_DFF=predict(fit_DFF,.),
                pred_DGS2=predict(fit_DGS2,.),
                pred_DGS10=predict(fit_DGS10,.),
                pred_MORTGAGE30US=predict(fit_MORTGAGE30US,.)) %>% 
  dplyr::select(date,pred_DFF:pred_MORTGAGE30US) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_DFF:pred_MORTGAGE30US) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>%
  filter(one_year) %>% 
  filter(date>=ymd("2000-01-01")) %>% 
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line(size=0.25) +
  scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  scale_y_continuous(breaks = seq(0,10,0.5)) +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  # facet_wrap(~one_year,scales="free",ncol=1) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted/Observed T-Bond Yields",
       x=NULL,
       y="(%)",
       caption = timestamp_caption()) +
  add_fomc_meeting_dates()

In [ ]:
df_predict_DFF %>% 
  dplyr::mutate(pred_spread=predict(fit_DGS10,.)-predict(fit_DGS2,.)) %>% 
  dplyr::select(date,pred_spread) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_spread) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>%
  filter(one_year) %>% 
  filter(date>=ymd("2000-01-01")) %>% 
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line(size=0.25) +
  scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  scale_y_continuous(breaks = seq(-10,10,0.5)) +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  # facet_wrap(~one_year,scales="free",ncol=1) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted/Observed Spread 2-10 T-Bond Yields",
       x=NULL,
       y="(%)",
       caption = timestamp_caption()) +
  add_fomc_meeting_dates()

## Future Yield Curves

In [ ]:
df_predict_DFF %>% 
  dplyr::mutate(pred_DFF=predict(fit_DFF,.),
                pred_DGS2=predict(fit_DGS2,.),
                pred_DGS10=predict(fit_DGS10,.),
                pred_MORTGAGE30US=predict(fit_MORTGAGE30US,.)) %>% 
  dplyr::select(date,pred_DFF:pred_MORTGAGE30US) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  filter(date>=Sys.Date()) %>% 
  filter(date %in% unique(date)[seq(1,length(date),length.out=6)]) %>%
  gather(pred,val,pred_DFF:pred_MORTGAGE30US) %>% 
  dplyr::mutate(pred=factor(pred,levels=c("pred_DFF",
                                          "pred_DGS2",
                                          "pred_DGS10",
                                          # "pred_DGS30",
                                          "pred_MORTGAGE30US"))) %>% 
  ggplot(.,aes(x=pred,y=val,color=factor(date),group=date)) +
  geom_line(size=1) +
  scale_y_continuous(breaks = seq(0,10,0.5)) +
  theme_money_printer_go_brrr(base_size=20) +
  labs(title="Predicted Yield Curves",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

## Future FED FOMC Meeting Rate changes

In [ ]:
df_predict_DFF %>% 
  dplyr::mutate(pred_DFF=predict(fit_DFF,.),
                pred_DGS2=predict(fit_DGS2,.),
                pred_DGS10=predict(fit_DGS10,.),
                pred_MORTGAGE30US=predict(fit_MORTGAGE30US,.)) %>% 
  dplyr::select(date,pred_DFF:pred_MORTGAGE30US) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_DFF:pred_MORTGAGE30US) %>%
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>% 
  tidyr::crossing(fomc_dates) %>% 
  group_by( date_fomc ) %>% 
  filter(date-date_fomc>=0) %>% 
  filter(date-date_fomc==min(date-date_fomc)) %>% 
  filter(pred=="pred_DFF") %>% 
  dplyr::mutate(val=ceiling(val/0.25)*0.25/100) %>% 
  ggplot(.,aes(x=date_fomc,y=val)) +
  geom_point(size=3) + 
  geom_line(size=0.25) +
  scale_y_continuous(breaks = seq(0,0.1,by=0.0025),labels = scales::percent,limits = c(0,NA)) +
  scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  geom_vline(data=fomc_dates,aes(xintercept=date_fomc),linetype=2) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted FOMC Meeting Rates",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

In [ ]:
fit_DFF_previous = rbindlist(lapply(c(7,30,180,365),function(days_ago) {
  
  temp_df =df_predict_DFF %>% 
    filter(date<=Sys.Date()-days(days_ago))
  
  temp_fit = caret::train(x = temp_df %>% 
                              na.omit %>% 
                              dplyr::select(-date,-value) %>% 
                              data.frame(),
                         y = temp_df %>% 
                           na.omit %>% 
                           .[["value"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
  
  temp_df %>% 
    dplyr::mutate(pred_DFF=predict(temp_fit,.)) %>% 
    dplyr::select(date,pred_DFF) %>% 
    dplyr::mutate(date=date+years(1),
                  days_ago=days_ago,) %>%
    gather(pred,val,pred_DFF) 
  
}))

fit_DFF_previous %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01"),
                days_ago=factor(days_ago)) %>% 
  tidyr::crossing(fomc_dates) %>% 
  group_by( date_fomc ) %>% 
  filter(date-date_fomc>=0) %>% 
  filter(date-date_fomc==min(date-date_fomc)) %>% 
  filter(pred=="pred_DFF") %>% 
  dplyr::mutate(val=ceiling(val/0.25)*0.25/100) %>% 
  ggplot(.,aes(x=date_fomc,y=val,color=days_ago)) +
  geom_point() + 
  geom_line() +
  scale_y_continuous(breaks = seq(0,0.1,by=0.0025),labels = scales::percent,limits = c(0,NA)) +
  scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  geom_vline(data=fomc_dates,aes(xintercept=date_fomc),linetype=2) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted FOMC Meeting Rates",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

## Danish Short and Long

In [ ]:
df_predict_DK_Short %>% 
  dplyr::mutate(pred_DK_Short=predict(fit_DK_Short,.),
                pred_DK_Long=predict(fit_DK_Long,.)) %>% 
  dplyr::mutate(pred_spread=pred_DK_Long-pred_DK_Short) %>% 
  dplyr::select(date,pred_DK_Short,pred_DK_Long,pred_spread,value) %>%
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_DK_Short:pred_spread) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01"))  %>%
    filter(!one_year) %>% 
  filter(date>=ymd("2000-01-01")) %>% 
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line() +
  geom_line(data=df_DK_Short %>% 
               dplyr::mutate(pred="Observed") %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")),aes(x=date,y=value))  +
  geom_line(data=df_DK_Long %>% 
               dplyr::mutate(pred="Observed") %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")),aes(x=date,y=value))  +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  scale_y_continuous(breaks = seq(0,10,0.5)) +
  theme(axis.text.x=element_text(angle=45,hjust=1))  +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted/Observed Long and Short Interest Rate",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

In [ ]:
df_predict_DK_Short %>% 
  dplyr::mutate(pred_DK_Short=predict(fit_DK_Short,.),
                pred_DK_Long=predict(fit_DK_Long,.)) %>% 
  dplyr::mutate(pred_spread=pred_DK_Long-pred_DK_Short) %>% 
  dplyr::select(date,pred_DK_Short,pred_DK_Long,pred_spread,value) %>%
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_DK_Short:pred_spread) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01"))  %>%
  filter(one_year) %>% 
  ggplot(.,aes(x=date,y=val,color=pred),size=2) +
  geom_line() +
  geom_line(data=df_DK_Short %>% 
               dplyr::mutate(pred="Observed") %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")),aes(x=date,y=value))  +
  geom_line(data=df_DK_Long %>% 
               dplyr::mutate(pred="Observed") %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")),aes(x=date,y=value))  +
  scale_x_date(date_breaks = "1 month",date_labels = "%b",limits=c(Sys.Date()-years(1),NA)) +
  scale_y_continuous(breaks = seq(0,10,0.5),limits=c(NA,7)) +
  theme(axis.text.x=element_text(angle=45,hjust=1))  +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted/Observed Long and Short Interest Rate",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

## USREC

In [ ]:
df_predict_UMCSENT %>% 
    dplyr::mutate(pred_UMCSENT=predict(fit_UMCSENT,.)) %>% 
  dplyr::select(date,pred_UMCSENT,value) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_UMCSENT:value) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>%
  # filter(one_year) %>% 
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line(size=1) +
  # scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  # scale_y_continuous(breaks = seq(0,10,0.5),limits=c(0,NA)) +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  add_fomc_meeting_dates() +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 2,linetype=2) +
  # geom_hline(yintercept = 0,linetype=2) +
  # facet_wrap(~one_year,scales="free",ncol=1) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Recession Prob",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

## USREC

In [ ]:
df_predict_USREC %>% 
    dplyr::mutate(pred_USREC=predict(fit_USREC,.)) %>% 
  dplyr::select(date,pred_USREC,value) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_USREC:value) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>%
  filter(one_year) %>%
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line(size=1) +
  # scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  # scale_y_continuous(breaks = seq(0,10,0.5),limits=c(0,NA)) +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  add_fomc_meeting_dates() +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 1,linetype=2) +
  # geom_hline(yintercept = 0,linetype=2) +
  # facet_wrap(~one_year,scales="free",ncol=1) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Recession Prob",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

## M2

In [ ]:
df_predict_M2SL %>% 
    dplyr::mutate(pred_M2SL=predict(fit_M2SL,.)) %>% 
  dplyr::select(date,pred_M2SL,value) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_M2SL:value) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>%
  # filter(one_year) %>% 
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line(size=1) +
  # scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  # scale_y_continuous(breaks = seq(0,10,0.5),limits=c(0,NA)) +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  add_fomc_meeting_dates() +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 2,linetype=2) +
  # geom_hline(yintercept = 0,linetype=2) +
  # facet_wrap(~one_year,scales="free",ncol=1) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted/Observed YoY M2",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

## Inflation Expectations

In [ ]:
df_predict_CPIAUCSL %>% 
    dplyr::mutate(pred_CPIAUCSL=predict(fit_CPIAUCSL,.)) %>% 
  dplyr::select(date,pred_CPIAUCSL,value) %>% 
  dplyr::mutate(date=date+years(1)) %>%
  gather(pred,val,pred_CPIAUCSL:value) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(date=ifelse(one_year,
                            ifelse(date>=Sys.Date()-years(1),date,NA),date)) %>% 
  filter(!is.na(date)) %>% 
  dplyr::mutate(date=as.Date(date,origin="1970-01-01")) %>%
  filter(one_year) %>% 
  ggplot(.,aes(x=date,y=val,color=pred)) +
  geom_line(size=1) +
  scale_x_date(date_breaks = "1 month",date_labels = "%b") +
  scale_y_continuous(breaks = seq(0,10,0.5),limits=c(0,NA)) +
  theme(axis.text.x=element_text(angle=45,hjust=1)) +
  add_fomc_meeting_dates() +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 2,linetype=2) +
  # geom_hline(yintercept = 0,linetype=2) +
  # facet_wrap(~one_year,scales="free",ncol=1) +
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Predicted/Observed YoY Inflation",
       x=NULL,
       y="(%)",
       caption = timestamp_caption())

## Previous Hiking Cycles

In [ ]:
fomc_decisions = do.call("rbind",list(
data.frame(type="hike",
           start_date=c("1994-02-04","1997-03-25","1999-06-30","2004-06-30","2015-12-07","2022-03-17")),
data.frame(type="cut",
           start_date=c("1990-07-13","1995-07-06","1998-09-29","2001-01-03","2002-11-06","2007-09-18","2008-10-08","2019-08-01","2020-03-03"))
)) %>% 
  dplyr::mutate(start_date=ymd(start_date))

# Crossing
df_DFF %>% 
  tidyr::crossing( fomc_decisions ) %>% 
  dplyr::mutate(date_diff=as.numeric(date-start_date)) %>% 
  filter( date_diff <= 365,
          date_diff >= -365) %>% 
  dplyr::mutate(start_date=factor(start_date)) %>% 
  ggplot(.,aes(x=date_diff,y=value,color=start_date,group=start_date)) +
  geom_line() +
  facet_wrap(~type)+
  theme_money_printer_go_brrr(base_size=20) 

# ECB Yield

In [ ]:
# Deposit Rate
df_ECB_DepositRate = ecb::get_data(key = glue("FM.D.U2.EUR.4F.KR.DFR.LEV"))   %>% 
  dplyr::mutate(obstime=ymd(obstime),
                Yield="0M") %>% 
    dplyr::select(Yield,obstime,obsvalue)
  
# Yields
yields = c( paste0(c(3,6,9),"M") , paste0(seq(30),"Y"))
df_ECB_yields = do.call("rbind",lapply(yields,function(yield) {
  ecb::get_data(key = glue("YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_{yield}"))  %>% 
  dplyr::mutate(obstime=ymd(obstime),
                Yield=yield) %>% 
    dplyr::select(Yield,obstime,obsvalue)
}))

# Combined
df_ECB_yields =
  df_ECB_DepositRate %>% 
  bind_rows(df_ECB_yields) %>% 
  dplyr::mutate(Yield_Numeric=as.integer(paste0(recode(Yield,
                               '0M'='0',
                               '3M'='3',
                               '6M'='6',
                               # '9M'='9',
                               '1Y'=paste0(glue("{1*12}")),
                               '2Y'=paste0(glue("{2*12}")),
                               # '3Y'=paste0(glue("{3*12}")),
                               # '4Y'=paste0(glue("{4*12}")),
                               '5Y'=paste0(glue("{5*12}")),
                               # '6Y'=paste0(glue("{6*12}")),
                               # '7Y'=paste0(glue("{7*12}")),
                               # '8Y'=paste0(glue("{8*12}")),
                               # '9Y'=paste0(glue("{9*12}")),
                               '10Y'=paste0(glue("{10*12}")),
                               # '11Y'=paste0(glue("{11*12}")),
                               # '12Y'=paste0(glue("{12*12}")),
                               # '13Y'=paste0(glue("{13*12}")),
                               # '14Y'=paste0(glue("{14*12}")),
                               # '15Y'=paste0(glue("{15*12}")),
                               # '16Y'=paste0(glue("{16*12}")),
                               # '17Y'=paste0(glue("{17*12}")),
                               # '18Y'=paste0(glue("{18*12}")),
                               # '19Y'=paste0(glue("{19*12}")),
                               # '20Y'=paste0(glue("{20*12}")),
                               # '21Y'=paste0(glue("{21*12}")),
                               # '22Y'=paste0(glue("{22*12}")),
                               # '23Y'=paste0(glue("{23*12}")),
                               # '24Y'=paste0(glue("{24*12}")),
                               # '25Y'=paste0(glue("{25*12}")),
                               # '26Y'=paste0(glue("{26*12}")),
                               # '27Y'=paste0(glue("{27*12}")),
                               # '28Y'=paste0(glue("{28*12}")),
                               # '29Y'=paste0(glue("{29*12}")),
                               '30Y'=paste0(glue("{30*12}")))))) %>% 
  dplyr::mutate(obsvalue=obsvalue/100)

## Yield Curve

In [ ]:
breaks_ = unique(df_ECB_yields$Yield_Numeric)
labels_ = unique(df_ECB_yields$Yield)

df_ECB_yields %>% 
  group_by(Yield) %>% 
  filter(obstime==max(obstime)) %>% 
  ggplot(.,aes(x=Yield_Numeric,y=obsvalue)) +
  geom_point() +
  geom_line() +
  # scale_x_continuous(labels = labels_,
  #                    breaks = breaks_) +
  scale_y_continuous(breaks = seq(-0.5,0.5,by=0.0025),labels = scales::percent)+
  theme_money_printer_go_brrr(base_size=12) +
  theme(axis.text.x=element_text(angle=45,hjust=1,size=8))

In [ ]:
checkboard_yields = c('1M','3M','6M','1Y','2Y','3Y','5Y','7Y','10Y','20Y','30Y')

df_checkboard = expand.grid(Yield=checkboard_yields,
                            Joined_Yield=checkboard_yields)

df_yield_checkboard = 
  df_checkboard %>% 
  inner_join(df_ECB_yields) %>% 
  inner_join(df_ECB_yields,by=c("Joined_Yield"="Yield","obstime"="obstime")) %>% 
  dplyr::mutate(spread=obsvalue.x-obsvalue.y) %>% 
  dplyr::mutate(label_=ifelse(spread<0,spread,NA)) %>% 
  dplyr::mutate(label_=scales::percent(x = label_,accuracy = 0.01,)) %>%
  dplyr::select(Yield,Joined_Yield,obstime,spread,label_) %>% 
  dplyr::mutate(Yield=factor(Yield,levels=checkboard_yields),
                Joined_Yield=factor(Joined_Yield,levels=checkboard_yields))

df_yield_checkboard %>% 
  group_by(Yield,Joined_Yield) %>% 
  filter(obstime==max(obstime)) %>% 
  ungroup() %>% 
  dplyr::mutate(spread=ifelse(Yield==Joined_Yield,NA,spread)) %>% 
  ggplot(.,aes(x=Yield,y=Joined_Yield)) +
  geom_tile(aes(fill=spread)) +
  scale_fill_gradient2(low="green",high="red",midpoint=0,na.value = "gray70") +
  theme_money_printer_go_brrr(base_size=12) +
  theme(axis.text.x=element_text(angle=45,hjust=1),
        legend.position="bottom")+
  geom_text(aes(label=label_),size=3.5)

## Spreads

### 2s10s

In [ ]:
df_ECB_yields %>% 
  dplyr::select(-Yield_Numeric) %>% 
  filter(Yield %in% c("2Y","10Y")) %>% 
  spread(Yield,obsvalue) %>% 
  dplyr::mutate(spread=(`10Y`-`2Y`)) %>% 
  ggplot(.,aes(x=obstime,y=spread)) +
  geom_line() +
  geom_hline(yintercept = 0,linetype=2) + 
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.1,1,by=0.0025)) + 
  theme_money_printer_go_brrr(base_size=12) +
  labs(title="Spread: 2Y - 10Y",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption()
       )

## Predict Yield

In [ ]:
num_trees=100
tune_grid=data.frame(mtry=c(3))

train_control = caret::trainControl(method = "repeatedcv",number = 5,repeats = 5)

df_DF = 
  df_ECB_yields %>% 
  filter(Yield=="0M") %>% 
  dplyr::select(obstime,obsvalue)

df_predict_DF = df_yield_checkboard %>% 
  filter(Yield!=Joined_Yield) %>% 
  dplyr::mutate(one_year_ahead=obstime+years(1)) %>% 
  left_join(df_DF,by=c("one_year_ahead"="obstime")) %>% 
  unite(Yield,Yield,Joined_Yield) %>% 
  dplyr::select(obstime,Yield,spread,obsvalue) %>% 
  dplyr::mutate(Yield=paste0("S",Yield)) %>% 
  spread(Yield,spread)

fit_DF = caret::train(x = df_predict_DF %>% 
                              na.omit %>% 
                              dplyr::select(-obstime,-obsvalue) %>% 
                              data.frame(),
                         y = df_predict_DF %>% 
                           na.omit %>% 
                           .[["obsvalue"]],
                         method = "rf",
                         tuneGrid=tune_grid,
                         ntree=num_trees,
                         metric = "RMSE",
                         trControl=train_control)
fit_DF

## Extrapolate

In [ ]:
df_predict_DF %>% 
  dplyr::mutate(pred_DF=predict(fit_DF,.)) %>% 
  dplyr::select(obstime,obsvalue,pred_DF) %>% 
  dplyr::mutate(obstime=obstime+years(1)) %>%
  gather(pred,val,obsvalue:pred_DF) %>%
  tidyr::crossing(one_year=c(TRUE,FALSE)) %>% 
  dplyr::mutate(obstime=ifelse(one_year,
                            ifelse(obstime>=Sys.Date()-years(1),obstime,NA),obstime)) %>% 
  filter(!is.na(obstime)) %>% 
  filter(one_year) %>% 
  dplyr::mutate(obstime=as.Date(obstime,origin="1970-01-01")) %>%
  ggplot(.,aes(x=obstime,y=val,color=pred)) +
  geom_line() +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  scale_y_continuous(labels = scales::percent) +
  theme_money_printer_go_brrr(base_size=12) +
  # theme(axis.text.x=element_text(angle=45,hjust=1)) +
  geom_vline(xintercept = Sys.Date(),linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  facet_wrap(~one_year,scales="free",ncol=1)

## ECB to Danish Yields

In [ ]:
df_DK10Y = ecb::get_data(key = "IRS.M.DK.L.L40.CI.0000.DKK.N.Z")  %>% 
  dplyr::mutate(obstime=ymd(paste0(obstime,"-01")),
                Yield="10Y",
                dk_obsvalue=obsvalue/100) %>% 
    dplyr::select(Yield,obstime,dk_obsvalue)

df_DK3MO = ecb::get_data(key = "FM.M.DK.DKK.DS.MM.CIBOR3M.ASKA")  %>% 
  dplyr::mutate(obstime=ymd(paste0(obstime,"-01")),
                Yield="3M",
                dk_obsvalue=obsvalue/100) %>% 
    dplyr::select(Yield,obstime,dk_obsvalue)

df_ECB_DK =
  df_ECB_yields %>% 
  filter(Yield %in% c("3M","10Y")) %>% 
  inner_join( bind_rows(df_DK10Y,df_DK3MO) ) %>% 
  dplyr::mutate(spread=dk_obsvalue-obsvalue) %>% 
  dplyr::select(-Yield_Numeric) %>% 
  gather(stat,val,obsvalue:spread) %>% 
  dplyr::mutate(stat=recode(stat,
                            'obsvalue'='EU 3 Mo.',
                            'spread'='DK-EU Spread',
                            'dk_obsvalue'='DK Interbank 3 Mo.'))


df_ECB_DK %>% 
  ggplot(.,aes(x=obstime,y=val,color=stat)) +
  geom_line() +
  facet_wrap(~Yield) +
  scale_y_continuous(labels = scales::percent,breaks = seq(-0.01,0.1,by=0.005)) +
  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +
  geom_hline(yintercept = 0,linetype=2) +
  theme_money_printer_go_brrr(base_size=12)  +
  labs(title="Spread: DK Interbank 3MO - EU 3 Month Bond",
       x="Date",
       y="Spread (%)",
       caption = timestamp_caption()
       )